In [2]:
# Cell 0 — Run ONCE, then comment out again
!pip install \
    langchain==1.2.17 \
    langchain-openai==1.2.1 \
    langchain-community==0.4.1 \
    langchain-pinecone==0.2.13 \
    pinecone \
    openai \
    pypdf \
    python-dotenv \
    tiktoken \
    pandas \
    tqdm

  Using cached langchain-1.2.17-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_openai-1.2.1-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached openai-2.36.0-py3-none-any.whl.metadata (31 kB)
  Using cached pypdf-6.10.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached tiktoken-0.12.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached pandas-3.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached langchain_core-1.3.3-py3-none-any.whl.metadata (4.5 kB)
  Using cached langgraph-1.1.10-py3-none-any.whl.metadata (8.0 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached langchain_classic-1.0.7-py3-none-any.whl.metadata (5.1 kB)
  Using cached sqlalchemy-2.0.49-cp312-cp312-macosx_11_0_arm64.whl.metadata (9.5 kB)
  U

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print("✓ Environment variables loaded")

In [ ]:
from pathlib import Path
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
import pinecone
import os
from dotenv import load_dotenv

load_dotenv()

pdf_dir = Path("data")
pdf_files = list(pdf_dir.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs: {[f.name for f in pdf_files]}")

documents = []
for pdf_path in pdf_files:
    reader = PdfReader(pdf_path)
    text = "".join([page.extract_text() for page in reader.pages])
    documents.append({
        "content": text,
        "source": pdf_path.name
    })

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

split_docs = []
for doc in documents:
    chunks = text_splitter.split_text(doc["content"])
    split_docs.extend([
        {"content": chunk, "source": doc["source"]} 
        for chunk in chunks
    ])

print(f"Created {len(split_docs)} chunks from PDFs")

pc_api_key = os.getenv("PINECONE_API_KEY")
pc_index_name = "trustworthy-ai"

pc = pinecone.Pinecone(api_key=pc_api_key)
if pc_index_name not in pc.list_indexes().names():
    pc.create_index(
        name=pc_index_name,
        dimension=1536,
        metric="cosine"
    )
    print(f"Created index: {pc_index_name}")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

texts = [doc["content"] for doc in split_docs]
metadatas = [{"source": doc["source"]} for doc in split_docs]

vector_store = PineconeVectorStore.from_texts(
    texts=texts,
    embedding=embeddings,
    index_name=pc_index_name,
    metadatas=metadatas
)

print(f"✓ Indexed {len(split_docs)} chunks in Pinecone")
print(f"Index: {pc_index_name}")